# Vorflow: Common Geometry Problems and How to Handle Them

This notebook is a practical guide for users building conceptual meshes with vorflow. Each section introduces a problem you are likely to encounter when loading real geometry — what it looks like, what goes wrong if you ignore it, and exactly which parameter or utility function solves it.

Work through the sections top to bottom. By the end you will know what to watch for before your first real mesh run.

In [ ]:
%load_ext autoreload
%autoreload 2

import time
from copy import deepcopy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import geopandas as gpd

from shapely.geometry import Point, Polygon, LineString, box
from shapely.ops import unary_union

from vorflow import ConceptualMesh, MeshGenerator, VoronoiTessellator, ThresholdField
from vorflow.utils import (
    calculate_mesh_quality,
    summarize_quality,
    check_geometry_resolution,
    resample_geometry,
)


In [ ]:
CRS = "EPSG:3857"


# ---------------------------------------------------------------------------
# Geometry helpers
# ---------------------------------------------------------------------------
def to_gdf(geometries, crs=CRS):
    return gpd.GeoDataFrame({"geometry": geometries}, crs=crs)


def vertex_count(geom):
    if geom.geom_type == "Polygon":
        total = len(geom.exterior.coords)
        for interior in geom.interiors:
            total += len(interior.coords)
        return total
    if geom.geom_type == "LineString":
        return len(geom.coords)
    if geom.geom_type == "MultiPolygon":
        return sum(vertex_count(part) for part in geom.geoms)
    return 0


def gdf_vertex_count(gdf):
    if gdf is None or gdf.empty:
        return 0
    return int(gdf.geometry.apply(vertex_count).sum())


# ---------------------------------------------------------------------------
# Pipeline runner
# ---------------------------------------------------------------------------
def run_case(
    name,
    domain_spec,
    line_specs=None,
    point_specs=None,
    polygon_specs=None,
    background_lc=1.0,
    cm_kwargs=None,
    mg_kwargs=None,
    gen_kwargs=None,
    resample_spacing=None,
):
    """Run a complete vorflow pipeline and return all artifacts for comparison."""
    cm_kwargs = cm_kwargs or {}
    mg_kwargs = mg_kwargs or {}
    gen_kwargs = gen_kwargs or {}
    line_specs = line_specs or []
    point_specs = point_specs or []
    polygon_specs = polygon_specs or []

    # Optional upstream resampling of non-domain polygons
    if resample_spacing:
        polygon_specs = [
            {**ps, "geometry": resample_geometry(ps["geometry"], resample_spacing)}
            for ps in polygon_specs
        ]

    # Raw GDFs (for later plotting — assembled AFTER any resampling)
    raw_poly_geoms = [domain_spec["geometry"]] + [ps["geometry"] for ps in polygon_specs]
    raw_line_geoms = [ls["geometry"] for ls in line_specs]
    raw_point_geoms = [ps["geometry"] for ps in point_specs]

    raw_polygons = gpd.GeoDataFrame({"geometry": raw_poly_geoms}, crs=CRS)
    raw_lines = gpd.GeoDataFrame({"geometry": raw_line_geoms}, crs=CRS) if raw_line_geoms else None
    raw_points = gpd.GeoDataFrame({"geometry": raw_point_geoms}, crs=CRS) if raw_point_geoms else None

    # ConceptualMesh
    cm = ConceptualMesh(crs=CRS, **cm_kwargs)
    cm.add_polygon(
        domain_spec["geometry"],
        zone_id=domain_spec["zone_id"],
        resolution=domain_spec.get("resolution"),
        dist_min=domain_spec.get("dist_min"),
        dist_max=domain_spec.get("dist_max"),
        densify=domain_spec.get("densify"),
        z_order=domain_spec.get("z_order", 0),
        simplify_tolerance=domain_spec.get("simplify_tolerance"),
        fields=domain_spec.get("fields"),
        embed=domain_spec.get("embed", True),
    )
    for ps in polygon_specs:
        cm.add_polygon(
            ps["geometry"],
            zone_id=ps["zone_id"],
            resolution=ps.get("resolution"),
            z_order=ps.get("z_order", 0),
            dist_min=ps.get("dist_min"),
            dist_max=ps.get("dist_max"),
            densify=ps.get("densify"),
            simplify_tolerance=ps.get("simplify_tolerance"),
            fields=ps.get("fields"),
            embed=ps.get("embed", True),
        )
    for ls in line_specs:
        cm.add_line(
            ls["geometry"],
            line_id=ls["line_id"],
            resolution=ls["resolution"],
            is_barrier=ls.get("is_barrier", False),
            dist_min=ls.get("dist_min"),
            dist_max=ls.get("dist_max"),
            fields=ls.get("fields"),
            embed=ls.get("embed", True),
            densify=ls.get("densify", True),
            simplify_tolerance=ls.get("simplify_tolerance"),
        )
    for ps in point_specs:
        cm.add_point(
            ps["geometry"],
            point_id=ps["point_id"],
            resolution=ps["resolution"],
            dist_min=ps.get("dist_min"),
            dist_max=ps.get("dist_max"),
            simplify_tolerance=ps.get("simplify_tolerance"),
        )

    clean_polys, clean_lines, clean_points = cm.generate()

    # Meshing
    t0 = time.time()
    mesh_success = False
    mg = grid = quality = None
    error = ""
    try:
        mg = MeshGenerator(background_lc=background_lc, **mg_kwargs)
        mg.generate(clean_polys, clean_lines, clean_points, **gen_kwargs)
        mesh_success = True
        vt = VoronoiTessellator(mg, cm, clip_to_boundary=True)
        grid = vt.generate()
        quality = calculate_mesh_quality(grid)
    except Exception as exc:
        error = str(exc)
    mesh_s = round(time.time() - t0, 2)

    return dict(
        name=name,
        raw_polygons=raw_polygons,
        raw_lines=raw_lines,
        raw_points=raw_points,
        clean_polys=clean_polys,
        clean_lines=clean_lines,
        clean_points=clean_points,
        mg=mg,
        grid=grid,
        quality=quality,
        mesh_success=mesh_success,
        mesh_s=mesh_s,
        error=error,
    )


# ---------------------------------------------------------------------------
# Summary table helper
# ---------------------------------------------------------------------------
def result_row(case):
    """Return a dict suitable for building a summary DataFrame."""
    return {
        "case": case["name"],
        "raw_poly_vertices": gdf_vertex_count(case["raw_polygons"]),
        "clean_poly_vertices": gdf_vertex_count(case["clean_polys"]),
        "mesh_success": case["mesh_success"],
        "grid_cells": len(case["grid"]) if case["grid"] is not None else 0,
        "mesh_s": case["mesh_s"],
        "error": case["error"],
    }


# ---------------------------------------------------------------------------
# Plot helpers
# ---------------------------------------------------------------------------
def _plot_case_row(axes, case, label, tint, nodes=None):
    """Fill a 3-axes row (raw | clean | grid) for one case."""
    ax_raw, ax_clean, ax_grid = axes

    # Raw
    if case["raw_polygons"] is not None and not case["raw_polygons"].empty:
        case["raw_polygons"].boundary.plot(ax=ax_raw, color="black", linewidth=0.8)
    if case["raw_lines"] is not None and not case["raw_lines"].empty:
        case["raw_lines"].plot(ax=ax_raw, color="steelblue", linewidth=1.5, zorder=3)
    if case["raw_points"] is not None and not case["raw_points"].empty:
        case["raw_points"].plot(ax=ax_raw, color="crimson", markersize=8, zorder=4)
    ax_raw.set_title(f"{label}\nInput geometry", fontsize=9, fontweight="bold")
    ax_raw.set_facecolor(tint)

    # Clean
    if case["clean_polys"] is not None and not case["clean_polys"].empty:
        case["clean_polys"].boundary.plot(ax=ax_clean, color="black", linewidth=0.8)
    if case["clean_lines"] is not None and not case["clean_lines"].empty:
        case["clean_lines"].plot(ax=ax_clean, color="steelblue", linewidth=1.5, zorder=3)
    if case["clean_points"] is not None and not case["clean_points"].empty:
        case["clean_points"].plot(ax=ax_clean, color="green", markersize=8, zorder=4)
    ax_clean.set_title("After preprocessing", fontsize=9)
    ax_clean.set_facecolor(tint)

    # Grid
    if case["grid"] is not None and not case["grid"].empty:
        case["grid"].plot(ax=ax_grid, alpha=0.45, edgecolor="black", linewidth=0.25)
    if nodes is not None:
        ax_grid.scatter(
            nodes[:, 0], nodes[:, 1],
            s=5, color="dimgray", alpha=0.35, zorder=3, linewidths=0,
        )
    ax_grid.set_title("Voronoi grid", fontsize=9)

    for ax in axes:
        ax.set_aspect("equal")
        ax.grid(alpha=0.2)


def plot_comparison(case_a, case_b, label_a="Case A", label_b="Case B",
                    nodes_a=None, nodes_b=None):
    """2 × 3 comparison plot: raw / clean / grid for two cases."""
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    _plot_case_row(axes[0], case_a, label_a, "#fff0f0", nodes=nodes_a)
    _plot_case_row(axes[1], case_b, label_b, "#f0fff0", nodes=nodes_b)
    plt.suptitle(
        "Rows compare two cases; columns show input, preprocessing, and final Voronoi grid",
        fontweight="bold",
        fontsize=11,
    )
    plt.tight_layout(rect=(0, 0, 1, 0.96))
    plt.show()


def plot_case(case, label="", nodes=None):
    """1 × 3 single-case plot: raw / clean / grid."""
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    _plot_case_row(axes, case, label, "#f5f5ff", nodes=nodes)
    plt.suptitle(label, fontweight="bold", fontsize=12)
    plt.tight_layout(rect=(0, 0, 1, 0.95))
    plt.show()


---
## Problem 1 — Duplicate and Near-Duplicate Vertices

**What you'll see:** You export river or boundary geometry from a GIS tool and two consecutive vertices in the same linestring share the exact same coordinate — or differ by a sub-nanometre floating-point rounding error.

**What goes wrong:** Gmsh treats a zero-length or near-zero line segment as a degenerate element and can reject the geometry during OCC line creation.

**How vorflow handles it:** The meshing path now has a guard at geometry transfer: `MeshGenerator` filters invalid and near-duplicate coordinates before creating OCC entities. That means mesh generation can succeed even when the displayed `clean_lines` were later densified and have more vertices than the raw input.

The diagnostic below therefore checks **short consecutive segments**, not just vertex counts. Vertex counts are still useful context, but after densification they are not proof that duplicates were removed.


In [ ]:
domain = {
    "geometry": box(0, 0, 10, 10),
    "zone_id": 1,
    "resolution": 4.0,
    "dist_max": 20.0,
}

line_with_duplicate = LineString([(1, 1), (5, 5), (5, 5), (9, 9)])
line_with_near_duplicate = LineString([(1, 8), (5, 8), (5 + 1e-9, 8), (9, 8)])


def short_segment_count(line, tol=1e-8):
    coords = list(line.coords)
    return sum(
        Point(coords[i]).distance(Point(coords[i + 1])) <= tol
        for i in range(len(coords) - 1)
    )


duplicate_case = run_case(
    name="duplicate-and-near-duplicate-line-vertices",
    domain_spec=domain,
    line_specs=[
        {
            "geometry": line_with_duplicate,
            "line_id": "duplicate_vertices",
            "resolution": 1.0,
        },
        {
            "geometry": line_with_near_duplicate,
            "line_id": "near_duplicate_vertices",
            "resolution": 1.0,
        },
    ],
    background_lc=4.0,
)

# Show duplicate/near-duplicate diagnostics. Clean vertex counts are reported
# only as context because line densification can add vertices after cleaning.
for label, raw_line, clean_line in [
    ("Line 1 ? exact duplicate at (5, 5)", line_with_duplicate, duplicate_case["clean_lines"].geometry.iloc[0]),
    ("Line 2 ? near-duplicate pair at ~(5, 8)", line_with_near_duplicate, duplicate_case["clean_lines"].geometry.iloc[1]),
]:
    print(label + ":")
    print(f"  Input vertices        : {len(list(raw_line.coords))}")
    print(f"  Clean vertices        : {len(list(clean_line.coords))}  (after optional densification)")
    print(f"  Raw short segments    : {short_segment_count(raw_line)}")
    print(f"  Clean short segments  : {short_segment_count(clean_line)}")
    print()

print(f"Mesh generation succeeded: {duplicate_case["mesh_success"]}")
print("Note: any remaining near-zero segment is filtered before Gmsh OCC creation.")


In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
ax_raw, ax_clean, ax_grid = axes

# --- raw geometry: highlight the problem coordinates ---
duplicate_case["raw_polygons"].boundary.plot(ax=ax_raw, color="black", linewidth=1)
duplicate_case["raw_lines"].plot(ax=ax_raw, color="steelblue", linewidth=1.5)

for x, y, lbl in [(5, 5, "exact duplicate"), (5, 8, "near-duplicate\n(1e-9 m gap)")]:
    ax_raw.plot(x, y, "o", color="red", markersize=18, alpha=0.75, zorder=5)
    ax_raw.annotate(
        lbl, xy=(x, y), xytext=(x + 1.0, y + 0.8),
        fontsize=8, color="darkred",
        arrowprops=dict(arrowstyle="->", color="darkred", lw=1.2),
    )

ax_raw.set_title("Input geometry  ✗\n(duplicate vertices highlighted in red)", fontsize=10, color="darkred")
ax_raw.set_facecolor("#fff0f0")

# --- cleaned geometry ---
duplicate_case["clean_polys"].boundary.plot(ax=ax_clean, color="black", linewidth=1)
duplicate_case["clean_lines"].plot(ax=ax_clean, color="steelblue", linewidth=1.5)
ax_clean.set_title("After ConceptualMesh preprocessing  ✓\n(duplicates removed)", fontsize=10, color="darkgreen")
ax_clean.set_facecolor("#f0fff0")

# --- resulting Voronoi grid with subtle Gmsh triangle-mesh nodes ---
grid = duplicate_case.get("grid")
if grid is not None and not grid.empty:
    grid.plot(ax=ax_grid, alpha=0.5, edgecolor="black", linewidth=0.25)
mg = duplicate_case.get("mg")
if mg is not None and mg.nodes is not None:
    ax_grid.scatter(
        mg.nodes[:, 0], mg.nodes[:, 1],
        s=5, color="dimgray", alpha=0.35, zorder=3, linewidths=0,
        label="mesh nodes",
    )
ax_grid.set_title("Voronoi grid + mesh nodes (grey dots)\n(mesh succeeds cleanly)", fontsize=10)

for ax in axes:
    ax.set_aspect("equal")
    ax.grid(alpha=0.2)

plt.suptitle("Problem 1: Duplicate Vertices — detected and removed automatically", fontweight="bold", fontsize=12)
plt.tight_layout()
plt.show()


## Problem 2 — Features That Don't Quite Reach the Domain Boundary

**What you'll see:** After digitising in a GIS, a monitoring well is snapped to the wrong layer and ends up 5 cm outside the domain boundary.

**Current behavior:** If the point cannot be snapped into the resolved domain, `ConceptualMesh.generate()` removes it from `clean_points`. That is safer than sending an outside point to Gmsh and later clipping away its Voronoi cell, but the practical effect is the same: the well has **zero influence on the grid**.

**What goes wrong:** `connectivity_tolerance` defaults to 1 mm. A gap of 5 cm is well outside that range, so the point is not snapped. During domain clipping it is removed before meshing.

**Fix — `connectivity_tolerance`:** Raise the tolerance to cover the gap. Any endpoint within that distance of the nearest feature vertex (domain boundary, line vertex) is snapped onto it before meshing.

```python
cm = ConceptualMesh(crs="EPSG:3857", connectivity_tolerance=0.1)  # snap within 10 cm
```

**Important caveats for real data:**

* `connectivity_tolerance` uses Shapely's `snap()` internally, so the well can only move to an **existing vertex** of the reference geometry. It does **not** snap to an arbitrary point along an edge.
* In this demo the river already has a vertex at `(0, 1.0)`, so the well has an exact snap target.
* `ConceptualMesh.generate()` runs in this order: **simplify → snap/connectivity → clip to domain → densify**.
* That means `simplify_tolerance` can remove the very vertex you needed as a snap target.
* It also means `densify=` does **not** help snapping, because densified vertices are added only after snapping has finished.
* If your target location lies on a long edge with no nearby vertex, `resample_geometry()` applied **before** `add_polygon()` or `add_line()` is usually the better tool.
* In those cases, **resampling without simplification** can be the safer option, because simplification may clean away the candidate vertices you needed for the snap.

The comparison below uses a river crossing the domain and a well 5 cm outside the domain near the river's entry point. The river is intentionally kept as coarse as the background mesh while the well requests much finer cells, so the snapped case creates a visibly tighter local node cluster. The plot includes a zoomed panel because the full-domain grid can make the boundary-local difference look subtler than it is.

* **Bad** (`tol=0.01`, 1 cm): well stays outside → removed from `clean_points` → no local refinement  
* **Good** (`tol=0.1`, 10 cm): well snaps to the river's entry vertex at `(0, 1.0)` → proper boundary node → fine local refinement becomes visible


In [ ]:
# Geometry: domain box, river line from boundary to boundary, well 5 cm outside domain
snap_domain = {"geometry": box(0, 0, 2, 2), "zone_id": 1, "resolution": 1.0}
snap_lines = [{
    "geometry": LineString([(0, 1.0), (2, 1.0)]),
    "line_id": "river",
    "resolution": 1.0,
    "dist_max": 0.10,
    "densify": False,
}]
snap_points = [{
    "geometry": Point(-0.05, 1.0),
    "point_id": "well",
    "resolution": 0.025,
    "dist_max": 0.55,
}]
# Note: well is 5 cm outside the domain at x = 0. The river starts exactly at (0, 1.0)
# on the domain boundary, providing the snap target when tolerance is large enough.
# The river uses background-scale sizing; the well is much finer so snapping is visually obvious.

snap_low = run_case(
    name="connectivity_tolerance=0.01 (gap not snapped)",
    domain_spec=snap_domain,
    line_specs=snap_lines,
    point_specs=snap_points,
    background_lc=1.0,
    cm_kwargs={"connectivity_tolerance": 0.01},   # 5 cm gap > 1 cm tol → no snap
)

snap_good = run_case(
    name="connectivity_tolerance=0.1 (gap snapped)",
    domain_spec=snap_domain,
    line_specs=snap_lines,
    point_specs=snap_points,
    background_lc=1.0,
    cm_kwargs={"connectivity_tolerance": 0.1},    # 5 cm gap < 10 cm tol → snaps to (0, 1.0)
)

from shapely.geometry import box as _sbox
_domain_geom = _sbox(0, 0, 2, 2)

def _pt_info(case):
    pts = case["clean_points"]
    if pts is None or pts.empty:
        return "no points"
    geom = pts.geometry.iloc[0]
    d = _domain_geom.exterior.distance(geom)
    status = "ON boundary" if d < 1e-9 else f"{d*100:.1f} cm OUTSIDE"
    return f"{geom}  →  {status}"

print(f"Low tolerance  — well after preprocessing: {_pt_info(snap_low)}")
print(f"Good tolerance — well after preprocessing: {_pt_info(snap_good)}")
print()
cells_low = len(snap_low["grid"]) if snap_low["grid"] is not None else 0
cells_good = len(snap_good["grid"]) if snap_good["grid"] is not None else 0
print(f"Low tolerance  — {cells_low} Voronoi cells  (well clipped away, no local point refinement)")
print(f"Good tolerance — {cells_good} Voronoi cells (well snaps to a boundary vertex and activates fine refinement)")
print()
print("Caveat: the river already contributes a vertex at (0, 1.0).")
print("The visible difference comes from the snapped well requesting much finer cells than the river/background mesh.")

In [ ]:
nodes_low = snap_low["mg"].nodes if snap_low["mg"] is not None else None
nodes_good = snap_good["mg"].nodes if snap_good["mg"] is not None else None


def _zoom_node_count(nodes, xlim=(-0.08, 0.55), ylim=(0.45, 1.55)):
    if nodes is None:
        return 0
    in_x = (nodes[:, 0] >= xlim[0]) & (nodes[:, 0] <= xlim[1])
    in_y = (nodes[:, 1] >= ylim[0]) & (nodes[:, 1] <= ylim[1])
    return int((in_x & in_y).sum())

print(f"Low tolerance zoom nodes : {_zoom_node_count(nodes_low)}")
print(f"Good tolerance zoom nodes: {_zoom_node_count(nodes_good)}")

fig, axes = plt.subplots(2, 4, figsize=(22, 10))

_plot_case_row(
    axes[0, :3],
    snap_low,
    "tol=0.01 -- well remains outside",
    "#fff0f0",
    nodes=nodes_low,
)
_plot_case_row(
    axes[1, :3],
    snap_good,
    "tol=0.10 -- well snaps to river vertex",
    "#f0fff0",
    nodes=nodes_good,
)

def _plot_snap_zoom(ax, case, nodes, title, tint):
    if case["grid"] is not None and not case["grid"].empty:
        case["grid"].plot(ax=ax, alpha=0.50, edgecolor="black", linewidth=0.30)
    if case["clean_lines"] is not None and not case["clean_lines"].empty:
        case["clean_lines"].plot(ax=ax, color="steelblue", linewidth=1.8, zorder=4)
    if case["clean_points"] is not None and not case["clean_points"].empty:
        case["clean_points"].plot(ax=ax, color="crimson", markersize=45, zorder=6)
    if nodes is not None:
        ax.scatter(nodes[:, 0], nodes[:, 1], s=10, color="dimgray", alpha=0.55, zorder=5, linewidths=0)
    ax.set_xlim(-0.08, 0.55)
    ax.set_ylim(0.45, 1.55)
    ax.set_aspect("equal")
    ax.grid(alpha=0.25)
    ax.set_facecolor(tint)
    ax.set_title(title, fontsize=9, fontweight="bold")

_plot_snap_zoom(
    axes[0, 3],
    snap_low,
    nodes_low,
    "Zoom: well is outside\nno fine boundary node cluster",
    "#fff0f0",
)
_plot_snap_zoom(
    axes[1, 3],
    snap_good,
    nodes_good,
    "Zoom: snapped well on boundary\nfine node cluster is active",
    "#f0fff0",
)

for ax in axes[:, 3]:
    ax.axvline(0, color="black", linewidth=0.8, alpha=0.6)
    ax.axhline(1.0, color="steelblue", linewidth=0.8, linestyle="--", alpha=0.7)

plt.suptitle(
    "Problem 2: connectivity_tolerance controls whether the outside well participates in the mesh",
    fontweight="bold",
    fontsize=12,
)
plt.tight_layout(rect=(0, 0, 1, 0.96))
plt.show()


---
## Problem 3 — Irregular Vertex Spacing: Useful Tool or Extra Constraint?

**What you'll see:** Source geometry can have clustered vertices in one part of a boundary and long, simple segments elsewhere. That does not mean Gmsh will leave the long segment unmeshed: Gmsh still discretizes curves from the active mesh-size field.

**The real question:** Does changing the source geometry improve the final mesh, or does it just add constraints that Gmsh did not need?

**Tools to compare:**

| Tool | How to use | What it does | Tradeoff |
|------|-----------|-------------|----------|
| No preprocessing | pass geometry as-is | Lets Gmsh choose mesh nodes from the size field | Acceptable when quality diagnostics are already good |
| `add_polygon(..., simplify_tolerance=1.0)` / `add_line(..., simplify_tolerance=1.0)` | blueprint parameter | Removes redundant/noisy source vertices without inserting new CAD points | Can erase intentional small features if the tolerance is too high |
| `resample_geometry(geom, spacing)` | before `add_polygon` / `add_line` | Redistributes vertices to near-uniform spacing | May move/remove original vertices and sharp source spacing patterns |
| `simplify_tolerance` + `densify` | blueprint parameters | Removes noisy clusters first, then caps long segments | Best when the source has both noise and long gaps |

The cells below use deliberately coarse meshes so local differences are visible, and diagnostics rather than assumptions. Both experiments use the same numeric source-cleanup parameters (`simplify_tolerance=1.0`, `resample_spacing=10.0`, and `densify=10.0` only after simplification). 3B keeps a bigger cell-size setting, but its polygon context receives the same cleanup operation as the matching 3A column.


In [ ]:
# ---------------------------------------------------------------------------
# Problem 3A: embedded polygon boundary with noisy clusters and long gaps
# ---------------------------------------------------------------------------
noisy_uneven_coords = [
    (10, 10), (11, 10.2), (11.5, 9.8), (12, 10.2), (15, 10),
    (50, 10), (85, 10), (88, 14), (90, 20), (90, 85),
    (85, 90), (50, 95), (15, 90), (10, 85), (10, 50), (10, 20),
]
noisy_uneven_poly = Polygon(noisy_uneven_coords)

spacing_domain = {
    "geometry": box(0, 0, 100, 100),
    "zone_id": 1,
    "resolution": 20.0,
}


def run_spacing_polygon_case(
    name,
    geometry,
    *,
    densify=False,
    simplify_tolerance=None,
    resample_spacing=None,
    cm_kwargs=None,
    mg_kwargs=None,
):
    geom = resample_geometry(geometry, resample_spacing) if resample_spacing else geometry
    return run_case(
        name=name,
        domain_spec=spacing_domain,
        polygon_specs=[
            {
                "geometry": geom,
                "zone_id": 2,
                "resolution": 10,
                "z_order": 1,
                "dist_max": 30.0,
                "densify": densify,
                "simplify_tolerance": simplify_tolerance,
            }
        ],
        background_lc=20.0,
        cm_kwargs=cm_kwargs,
        mg_kwargs=mg_kwargs,
    )


polygon_spacing_cases = [
    run_spacing_polygon_case("polygon-baseline", noisy_uneven_poly, densify=False),
    run_spacing_polygon_case("polygon-simplify-1", noisy_uneven_poly, densify=False, simplify_tolerance=1.0),
    run_spacing_polygon_case("polygon-resample-10", noisy_uneven_poly, densify=False, resample_spacing=10.0),
    run_spacing_polygon_case(
        "polygon-simplify-1-densify-10",
        noisy_uneven_poly,
        densify=10.0,
        simplify_tolerance=1.0,
    ),
]


# ---------------------------------------------------------------------------
# Problem 3B: embedded diagonal line crossing the polygon context
# ---------------------------------------------------------------------------
spacing_line = LineString([(5, 5), (20, 20), (35, 35), (50, 50), (65, 65), (80, 80), (95, 95)])
spacing_line_domain = {
    "geometry": box(0, 0, 100, 100),
    "zone_id": 1,
    "resolution": 20.0,
}


def run_spacing_line_case(
    name,
    *,
    densify=False,
    simplify_tolerance=None,
    resample_spacing=None,
    polygon_densify=False,
    polygon_simplify_tolerance=None,
    polygon_resample_spacing=None,
    cm_kwargs=None,
    mg_kwargs=None,
    gen_kwargs=None,
):
    geom = resample_geometry(spacing_line, resample_spacing) if resample_spacing else spacing_line
    polygon_geom = resample_geometry(noisy_uneven_poly, polygon_resample_spacing) if polygon_resample_spacing else noisy_uneven_poly
    field = ThresholdField(
        size_min=5.0,
        dist_min=2.0,
        dist_max=18.0,
        size_max=20.0,
        sampling=5,
    )
    return run_case(
        name=name,
        domain_spec=spacing_line_domain,
        polygon_specs=[
            {
                "geometry": polygon_geom,
                "zone_id": 2,
                "resolution": 10.0,
                "z_order": 1,
                "dist_max": 30,
                "densify": polygon_densify,
                "simplify_tolerance": polygon_simplify_tolerance,
            }
        ],
        line_specs=[
            {
                "geometry": geom,
                "line_id": "embedded_diagonal_refinement_line",
                "resolution": 5.0,
                "fields": [field],
                "embed": True,
                "densify": densify,
                "simplify_tolerance": simplify_tolerance,
            }
        ],
        background_lc=20.0,
        cm_kwargs=cm_kwargs,
        mg_kwargs=mg_kwargs,
        gen_kwargs=gen_kwargs,
    )


spacing_line_cases = [
    run_spacing_line_case("line-polygon-baseline", densify=False, polygon_densify=False),
    run_spacing_line_case("line-polygon-simplify-1", simplify_tolerance=1.0, polygon_simplify_tolerance=1.0),
    run_spacing_line_case("line-polygon-resample-10", resample_spacing=10.0, polygon_resample_spacing=10.0),
    run_spacing_line_case(
        "line-polygon-simplify-1-densify-10",
        simplify_tolerance=1.0,
        densify=10.0,
        polygon_simplify_tolerance=1.0,
        polygon_densify=10.0,
    ),
]


In [ ]:
def _zone_geom(case, zone_id=2):
    zone_mask = case["clean_polys"]["zone_id"] == zone_id
    return case["clean_polys"].loc[zone_mask, "geometry"].iloc[0]


def _segment_stats(geom):
    stats = check_geometry_resolution(to_gdf([geom]))
    if isinstance(stats, str):
        return {"min": np.nan, "max": np.nan, "median": np.nan, "count": 0, "max_median_ratio": np.nan}
    median = stats["median"]
    return {
        "min": stats["min"],
        "max": stats["max"],
        "median": median,
        "count": stats["count"],
        "max_median_ratio": stats["max"] / median if median else np.nan,
    }


def _local_quality(case, reference_geom, band_distance):
    quality = case["quality"]
    if quality is None or quality.empty:
        return {"local_cells": 0, "compact_p05": np.nan, "compact_median": np.nan, "drift_p95": np.nan, "drift_max": np.nan, "area_cv": np.nan}

    generators = gpd.GeoSeries(gpd.points_from_xy(quality.x, quality.y), crs=CRS)
    target = reference_geom.boundary if reference_geom.geom_type in ("Polygon", "MultiPolygon") else reference_geom
    local = quality.loc[generators.distance(target) <= band_distance]
    if local.empty:
        return {"local_cells": 0, "compact_p05": np.nan, "compact_median": np.nan, "drift_p95": np.nan, "drift_max": np.nan, "area_cv": np.nan}

    return {
        "local_cells": int(len(local)),
        "compact_p05": local["compactness"].quantile(0.05),
        "compact_median": local["compactness"].median(),
        "drift_p95": local["drift_ratio"].quantile(0.95),
        "drift_max": local["drift_ratio"].max(),
        "area_cv": local["area"].std() / local["area"].mean(),
    }


def _vertex_context_row(label, geom, note):
    spacing = _segment_stats(geom)
    coords = list(geom.exterior.coords) if geom.geom_type == "Polygon" else list(geom.coords)
    return {
        "case": label,
        "vertices": len(coords),
        "min_seg": spacing["min"],
        "max_seg": spacing["max"],
        "median_seg": spacing["median"],
        "max_median_ratio": spacing["max_median_ratio"],
        "note": note,
    }


def _polygon_diagnostic_row(case):
    clean_geom = _zone_geom(case)
    spacing = _segment_stats(clean_geom)
    local = _local_quality(case, clean_geom, band_distance=15.0)
    return {
        "case": case["name"],
        "clean_vertices": len(clean_geom.exterior.coords),
        "clean_max_seg": spacing["max"],
        "max_median_ratio": spacing["max_median_ratio"],
        "cells": len(case["grid"]) if case["grid"] is not None else 0,
        **local,
    }


def _line_diagnostic_row(case):
    line_geom = case["clean_lines"].geometry.iloc[0]
    spacing = _segment_stats(line_geom)
    local = _local_quality(case, line_geom, band_distance=6.0)
    return {
        "case": case["name"],
        "clean_vertices": len(line_geom.coords),
        "clean_max_seg": spacing["max"],
        "cells": len(case["grid"]) if case["grid"] is not None else 0,
        **local,
    }


polygon_vertex_variants = [
    ("Baseline input", noisy_uneven_poly, "clustered vertices plus long gaps"),
    ("simplify=1.0", _zone_geom(polygon_spacing_cases[1]), "removes small wiggles without adding vertices"),
    ("resample_geometry(..., 10.0)", _zone_geom(polygon_spacing_cases[2]), "redistributes the ring to near-uniform spacing"),
    ("simplify=1.0 + densify=10.0", _zone_geom(polygon_spacing_cases[3]), "removes small wiggles, then caps long segments"),
]

vertex_context_df = pd.DataFrame([_vertex_context_row(label, geom, note) for label, geom, note in polygon_vertex_variants])
print("Source/clean vertex spacing context:")
display(vertex_context_df.round({"min_seg": 2, "max_seg": 2, "median_seg": 2, "max_median_ratio": 2}))

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
for ax, (label, geom, note) in zip(axes.ravel(), polygon_vertex_variants):
    coords = list(geom.exterior.coords)
    xs, ys = zip(*coords)
    ax.plot(xs, ys, color="black", linewidth=1.2)
    ax.plot(xs, ys, "o", color="red", markersize=2, zorder=5)
    ax.set_title(f"{label}\n{len(coords)} vertices - {note}", fontsize=10)
    ax.set_aspect("equal")
    ax.grid(alpha=0.2)
    ax.set_xlim(5, 95)
    ax.set_ylim(5, 100)
plt.suptitle("Problem 3A: source vertices before judging mesh quality", fontweight="bold", fontsize=12)
plt.tight_layout(rect=(0, 0, 1, 0.96))
plt.show()

polygon_diagnostics = pd.DataFrame([_polygon_diagnostic_row(case) for case in polygon_spacing_cases])
line_diagnostics = pd.DataFrame([_line_diagnostic_row(case) for case in spacing_line_cases])

print("Metric guide: compact_p05/compact_median higher is better; drift_p95/drift_max and area_cv lower are better; clean_max_seg is source spacing, not a quality score.")
print("Experiment 3A - embedded polygon boundary")
display(polygon_diagnostics.round({"clean_max_seg": 2, "max_median_ratio": 2, "compact_p05": 4, "compact_median": 4, "drift_p95": 4, "drift_max": 4, "area_cv": 4}))

print("Experiment 3B - embedded diagonal line crossing the polygon")
display(line_diagnostics.round({"clean_max_seg": 2, "compact_p05": 4, "compact_median": 4, "drift_p95": 4, "drift_max": 4, "area_cv": 4}))

print("Interpretation:")
print("  * Gmsh meshes long curves from the active size field; source densification is not mandatory.")
print("  * Simplify-only removes redundant/noisy source detail without adding new CAD points.")
print("  * Resampling redistributes source vertices; simplify + densify removes noise first, then caps long remaining segments.")
print("  * In 3B, the same cleanup is applied to the polygon context and the embedded line, while keeping the coarser cell-size settings.")
print("  * Use the least intrusive tool that improves the local diagnostics you care about; baseline is acceptable when metrics are already good.")

fig, axes = plt.subplots(2, 4, figsize=(22, 11), constrained_layout=True)
quality_metric = "compactness"
quality_vmin = 0.55
quality_vmax = 0.95
quality_cmap = "RdYlGn"


def _plot_quality_mesh(ax, case, title, overlay_geoms, xlim, ylim):
    quality = case["quality"]
    if quality is not None and not quality.empty:
        quality.plot(
            ax=ax,
            column=quality_metric,
            cmap=quality_cmap,
            vmin=quality_vmin,
            vmax=quality_vmax,
            edgecolor="black",
            linewidth=0.25,
            legend=False,
        )
    if not isinstance(overlay_geoms, (list, tuple)):
        overlay_geoms = [overlay_geoms]
    for geom, color, linewidth, linestyle in overlay_geoms:
        gpd.GeoSeries([geom], crs=CRS).plot(
            ax=ax,
            color=color,
            linewidth=linewidth,
            linestyle=linestyle,
            zorder=4,
        )
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_title(title, fontsize=10, fontweight="bold")
    ax.set_aspect("equal")
    ax.grid(alpha=0.18)


polygon_plot_cases = [
    (polygon_spacing_cases[0], "Polygon: baseline"),
    (polygon_spacing_cases[1], "Polygon: simplify=1"),
    (polygon_spacing_cases[2], "Polygon: resample=10"),
    (polygon_spacing_cases[3], "Polygon: simplify=1 + densify=10"),
]
for ax, (case, title) in zip(axes[0], polygon_plot_cases):
    _plot_quality_mesh(ax, case, title, [(_zone_geom(case).boundary, "black", 1.3, "-")], xlim=(0, 100), ylim=(0, 100))

line_plot_cases = [
    (spacing_line_cases[0], "Line + polygon: baseline"),
    (spacing_line_cases[1], "Line + polygon: simplify=1"),
    (spacing_line_cases[2], "Line + polygon: resample=10"),
    (spacing_line_cases[3], "Line + polygon: simplify=1 + densify=10"),
]
for ax, (case, title) in zip(axes[1], line_plot_cases):
    line_overlays = [
        (_zone_geom(case).boundary, "black", 1.1, "-"),
        (case["clean_lines"].geometry.iloc[0], "tab:blue", 1.8, "--"),
    ]
    _plot_quality_mesh(ax, case, title, line_overlays, xlim=(0, 100), ylim=(0, 100))

norm = mpl.colors.Normalize(vmin=quality_vmin, vmax=quality_vmax)
sm = mpl.cm.ScalarMappable(norm=norm, cmap=quality_cmap)
sm.set_array([])
fig.colorbar(
    sm,
    ax=axes.ravel().tolist(),
    orientation="horizontal",
    shrink=0.72,
    pad=0.08,
    aspect=40,
    label="Cell compactness (same scale for all panels; higher is better)",
)
fig.suptitle("Problem 3: Voronoi cell quality comparison with a shared color scale", fontweight="bold", fontsize=12)
plt.show()


---
## Problem 4 - Snapping, Healing, and Sliver Side Effects

**What you'll see:** A clean set of embedded lines is hard but meshable without healing: a horizontal line, a 5 degree line pinned at its midpoint, and a nearby vertical line. Then we add a tiny unintended polygon sliver near that same intersection cluster. The sliver creates extra local topology that `connectivity_tolerance` does not remove.

**Why this matters:** `connectivity_tolerance` is a preprocessing snap. In the current workflow it snaps line endpoints to polygon boundaries and points to nearby geometry; it does not solve interior line-line intersections or remove tiny embedded polygon slivers. Those are left for Gmsh/OCC fragmentation and, optionally, OCC healing.

**Knobs compared:**

```python
ConceptualMesh(connectivity_tolerance=...)
MeshGenerator(heal_shapes=True, heal_tolerance=...)
```

> **Rule of thumb:** First check whether the clean geometry meshes without healing. Use healing for real tiny artifacts such as slivers, and judge it by cost, mesh quality, and constraint preservation. A lower cell count is only useful if the embedded features are still represented and the quality metrics remain acceptable.


In [ ]:
pinned_domain = {
    "geometry": box(0, 0, 2, 2),
    "zone_id": 1,
    "resolution": 0.45,
}

pinned_angle_deg = 5.0
vertical_offset = 1e-4
pinned_line_resolution = 0.04
pinned_background_lc = 0.45
pinned_center = (1.0, 1.0)
pinned_half_length = 0.75

sliver_width = 1e-3
sliver_length = 0.05
sliver_origin = (1.00018, 1.00002)

theta = np.deg2rad(pinned_angle_deg)
dx = np.cos(theta) * pinned_half_length
dy = np.sin(theta) * pinned_half_length
cx, cy = pinned_center

pinned_line_geoms = [
    ("horizontal", LineString([(0.25, cy), (1.75, cy)])),
    (f"angled_{pinned_angle_deg:g}deg", LineString([(cx - dx, cy - dy), (cx + dx, cy + dy)])),
    ("near_vertical", LineString([(cx + vertical_offset, 0.35), (cx + vertical_offset, 1.65)])),
]

pinned_line_specs = [
    {
        "geometry": geom,
        "line_id": name,
        "resolution": pinned_line_resolution,
        "is_barrier": False,
        "dist_max": 0.12,
        "embed": True,
        "densify": True,
    }
    for name, geom in pinned_line_geoms
]

sliver_x, sliver_y = sliver_origin
sliver_polygon = Polygon(
    [
        (sliver_x, sliver_y - sliver_width / 2),
        (sliver_x + sliver_length, sliver_y - sliver_width / 2),
        (sliver_x + sliver_length, sliver_y + sliver_width / 2),
        (sliver_x, sliver_y + sliver_width / 2),
    ]
)
sliver_specs = [
    {
        "geometry": sliver_polygon,
        "zone_id": 2,
        "resolution": pinned_line_resolution,
        "z_order": 2,
        "embed": True,
        "densify": False,
    }
]

# At the vertical line, the horizontal and angled-line intersections are this far apart.
local_intersection_gap = abs(np.tan(theta) * vertical_offset)
constraint_node_tolerance = max(local_intersection_gap / 4, 1e-7)
minimum_nodes_per_line = 10

robustness_settings = [
    {
        "name": "clean-lines",
        "label": "Clean lines, no healing",
        "has_sliver": False,
        "polygon_specs": [],
        "mg_kwargs": {"tolerance_initial_delaunay": 1e-8, "heal_shapes": False, "optimization_cycles": 0, "smoothing_steps": 0},
    },
    {
        "name": "sliver-no-healing",
        "label": "Sliver, no healing",
        "has_sliver": True,
        "polygon_specs": sliver_specs,
        "mg_kwargs": {"tolerance_initial_delaunay": 1e-8, "heal_shapes": False, "optimization_cycles": 0, "smoothing_steps": 0},
    },
    {
        "name": "sliver-mild-healing",
        "label": "Sliver, healing tol=5e-5",
        "has_sliver": True,
        "polygon_specs": sliver_specs,
        "mg_kwargs": {"tolerance_initial_delaunay": 1e-8, "heal_shapes": True, "heal_tolerance": 5e-5, "optimization_cycles": 0, "smoothing_steps": 0},
    },
    {
        "name": "sliver-over-healing",
        "label": "Sliver, healing tol=1e-3",
        "has_sliver": True,
        "polygon_specs": sliver_specs,
        "mg_kwargs": {"tolerance_initial_delaunay": 1e-8, "heal_shapes": True, "heal_tolerance": 1e-3, "optimization_cycles": 0, "smoothing_steps": 0},
    },
]


def _constraint_node_counts(case, source_lines, distance_tolerance):
    if not case["mesh_success"] or case["mg"] is None or case["mg"].nodes is None:
        return [0 for _ in source_lines]
    nodes = case["mg"].nodes
    counts = []
    for _, line in source_lines:
        counts.append(int(sum(line.distance(Point(x, y)) <= distance_tolerance for x, y in nodes)))
    return counts


print(
    "Pinned embedded-line geometry: "
    f"angle={pinned_angle_deg:g} deg, vertical offset={vertical_offset:g}, "
    f"local intersection gap={local_intersection_gap:.2e}"
)
print(
    "Sliver artifact: "
    f"width={sliver_width:g}, length={sliver_length:g}, origin=({sliver_x:g}, {sliver_y:g})"
)
print(
    "Constraint preservation check: "
    f"at least {minimum_nodes_per_line} nodes per line within {constraint_node_tolerance:.2e} units"
)
print("Topology note: connectivity_tolerance is held at 1e-12; it does not remove this interior sliver or line-line cluster.")

robustness_cases = []
for setting in robustness_settings:
    print(f"Running {setting['label']}...")
    case = run_case(
        name=setting["name"],
        domain_spec=pinned_domain,
        polygon_specs=setting["polygon_specs"],
        line_specs=pinned_line_specs,
        background_lc=pinned_background_lc,
        cm_kwargs={"connectivity_tolerance": 1e-12},
        mg_kwargs=setting["mg_kwargs"],
    )
    node_counts = _constraint_node_counts(case, pinned_line_geoms, constraint_node_tolerance)
    case["label"] = setting["label"]
    case["has_sliver"] = setting["has_sliver"]
    case["settings"] = setting["mg_kwargs"]
    case["constraint_node_counts"] = node_counts
    case["constraint_preserved"] = case["mesh_success"] and min(node_counts) >= minimum_nodes_per_line
    robustness_cases.append(case)
    cells = len(case["grid"]) if case["grid"] is not None else 0
    print(
        f"  mesh_success={case['mesh_success']}  constraints_preserved={case['constraint_preserved']}  "
        f"cells={cells}  node_counts={node_counts}"
    )


def _robustness_row(case):
    quality = case["quality"]
    node_counts = case["constraint_node_counts"]
    lost_lines = sum(count < minimum_nodes_per_line for count in node_counts)
    settings = case["settings"]
    base = {
        "case": case["name"],
        "has_sliver": case["has_sliver"],
        "heal_tolerance": settings.get("heal_tolerance", np.nan),
        "mesh_success": case["mesh_success"],
        "constraints_preserved": case["constraint_preserved"],
        "lost_lines": lost_lines,
        "node_counts": str(node_counts),
        "min_nodes_per_line": min(node_counts),
        "cells": len(case["grid"]) if case["grid"] is not None else 0,
        "mesh_s": case["mesh_s"],
        "error": case["error"][:80],
    }
    if quality is None or quality.empty:
        return {**base, "compact_median": np.nan, "area_cv": np.nan}
    return {
        **base,
        "compact_median": quality["compactness"].median(),
        "area_cv": quality["area"].std() / quality["area"].mean(),
    }


robustness_diagnostics = pd.DataFrame([_robustness_row(case) for case in robustness_cases])
print(
    "Metric guide: mesh_success should be True and constraints_preserved should also be True. "
    "Fewer cells is only better when constraints are preserved. compact_median higher is better; area_cv lower is better."
)
display(robustness_diagnostics.round({"heal_tolerance": 8, "mesh_s": 2, "compact_median": 4, "area_cv": 4}))

sliver_no_healing_cells = int(robustness_diagnostics.loc[robustness_diagnostics["case"] == "sliver-no-healing", "cells"].iloc[0])
sliver_mild_healing_cells = int(robustness_diagnostics.loc[robustness_diagnostics["case"] == "sliver-mild-healing", "cells"].iloc[0])
sliver_mild_ok = bool(robustness_diagnostics.loc[robustness_diagnostics["case"] == "sliver-mild-healing", "constraints_preserved"].iloc[0])
print(
    "Mild healing comparison: "
    f"sliver case changes from {sliver_no_healing_cells} to {sliver_mild_healing_cells} cells "
    f"while constraints_preserved={sliver_mild_ok}. Inspect compactness/area_cv before accepting that tradeoff."
)

if robustness_diagnostics["constraints_preserved"].all():
    print("Result: every setting preserved the embedded lines; compare cell cost and quality.")
elif robustness_diagnostics["mesh_success"].all():
    print("Result: every setting meshed, but over-healing lost at least one embedded-line constraint.")
else:
    print("Result: at least one setting failed before producing a mesh.")


In [ ]:
fig, axes = plt.subplots(
    2,
    4,
    figsize=(20, 9),
    gridspec_kw={"height_ratios": [3.0, 2.1]},
    constrained_layout=True,
)
map_axes = axes[0]
zoom_axes = axes[1]
quality_metric = "compactness"
quality_vmin = 0.55
quality_vmax = 0.95
quality_cmap = "RdYlGn"
line_colors = {
    "horizontal": "#d73027",
    f"angled_{pinned_angle_deg:g}deg": "#4575b4",
    "near_vertical": "#111111",
}
sliver_face = "#fdae61"
sliver_edge = "#7f2704"


def _plot_problem4_lines(ax, linewidth=1.8):
    for name, geom in pinned_line_geoms:
        gpd.GeoSeries([geom], crs=CRS).plot(
            ax=ax,
            color=line_colors[name],
            linewidth=linewidth,
            zorder=6,
        )
    ax.scatter([cx], [cy], s=20, color="white", edgecolor="black", linewidth=0.7, zorder=7)


def _plot_problem4_sliver(ax, case):
    if not case["has_sliver"]:
        return
    gpd.GeoSeries([sliver_polygon], crs=CRS).plot(
        ax=ax,
        facecolor=sliver_face,
        edgecolor=sliver_edge,
        alpha=0.45,
        linewidth=1.0,
        zorder=5,
    )


def _plot_case_quality(ax, case, linewidth):
    quality = case["quality"]
    if quality is not None and not quality.empty:
        quality.plot(
            ax=ax,
            column=quality_metric,
            cmap=quality_cmap,
            vmin=quality_vmin,
            vmax=quality_vmax,
            edgecolor="black",
            linewidth=linewidth,
            legend=False,
        )
    else:
        gpd.GeoSeries([pinned_domain["geometry"]], crs=CRS).boundary.plot(
            ax=ax,
            color="black",
            linewidth=1.0,
        )
        ax.text(
            0.04,
            0.94,
            "mesh failed",
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=10,
            fontweight="bold",
            color="#b2182b",
            bbox={"facecolor": "white", "edgecolor": "#b2182b", "alpha": 0.85, "pad": 2},
        )


for ax_map, ax_zoom, case in zip(map_axes, zoom_axes, robustness_cases):
    _plot_case_quality(ax_map, case, linewidth=0.22)
    _plot_problem4_sliver(ax_map, case)
    _plot_problem4_lines(ax_map, linewidth=1.5)
    status = "OK" if case["mesh_success"] and case["constraint_preserved"] else "FAIL"
    cells = len(case["grid"]) if case["grid"] is not None else 0
    min_nodes = min(case["constraint_node_counts"])
    heal_label = "no healing" if not case["settings"].get("heal_shapes") else f"heal={case['settings']['heal_tolerance']:.0e}"
    ax_map.set_title(
        f"{case['name']}\n{status}, {cells} cells, min nodes/line={min_nodes}\n{heal_label}",
        fontsize=8.5,
        fontweight="bold",
    )
    ax_map.set_xlim(0, 2)
    ax_map.set_ylim(0, 2)
    ax_map.set_aspect("equal")
    ax_map.grid(alpha=0.18)
    ax_map.set_xlabel("")
    ax_map.set_ylabel("")

    _plot_case_quality(ax_zoom, case, linewidth=0.45)
    _plot_problem4_sliver(ax_zoom, case)
    _plot_problem4_lines(ax_zoom, linewidth=2.1)
    if case["mesh_success"] and case["mg"] is not None and case["mg"].nodes is not None:
        nodes = case["mg"].nodes
        window = (
            (nodes[:, 0] >= 0.94)
            & (nodes[:, 0] <= 1.08)
            & (nodes[:, 1] >= 0.965)
            & (nodes[:, 1] <= 1.035)
        )
        ax_zoom.scatter(nodes[window, 0], nodes[window, 1], s=8, color="black", alpha=0.65, zorder=8)
    ax_zoom.set_xlim(0.94, 1.08)
    ax_zoom.set_ylim(0.965, 1.035)
    ax_zoom.set_aspect("equal")
    ax_zoom.grid(alpha=0.25)
    ax_zoom.set_xlabel("intersection/sliver zoom")
    ax_zoom.set_ylabel("")

norm = mpl.colors.Normalize(vmin=quality_vmin, vmax=quality_vmax)
sm = mpl.cm.ScalarMappable(norm=norm, cmap=quality_cmap)
sm.set_array([])
fig.colorbar(
    sm,
    ax=map_axes.ravel().tolist(),
    orientation="horizontal",
    shrink=0.78,
    pad=0.03,
    aspect=36,
    label="Cell compactness (same range for all map panels; higher is better)",
)
fig.suptitle(
    "Problem 4: topology snapping is not sliver healing, and over-healing can lose constraints",
    fontweight="bold",
    fontsize=12,
)
plt.show()


---
## Known Limitations

The following cases show where the vorflow preprocessing pipeline **cannot** fully infer user intent — either because the problem is fundamentally topological, or because any automatic fix could destroy intended geometry.

---
### Limitation 1 — Polygon Holes Mean "Not This Zone," Not Automatically "Empty Void"

**What you'll see:** You create a polygon with a hole (donut shape) in zone 2 and place it inside zone 1.

**What vorflow can know:** The hole is definitely not zone 2. Vorflow cannot infer whether you intended that hole to be empty space, lower-priority zone 1, or a separate material.

**Current effect:** If a lower-priority polygon exists underneath, cells inside the hole are assigned to that lower-priority zone. In this demo the hole belongs to zone 1. That is valid behavior, and some examples/tests rely on it.

**Choose the geometry that matches your intent:**

```python
# 1) Hole should belong to the lower-priority/background zone:
#    use the donut polygon as-is. This is the current demo behavior.
cm.add_polygon(domain, zone_id=1, z_order=0)
cm.add_polygon(zone2_with_hole, zone_id=2, z_order=1)

# 2) Hole should be empty / outside the mesh:
#    remove the hole from the underlying domain too.
domain_with_void = domain.difference(hole_polygon)
cm.add_polygon(domain_with_void, zone_id=1, z_order=0)
cm.add_polygon(zone2_with_hole, zone_id=2, z_order=1)

# 3) Hole should be a third material/zone:
#    add it explicitly as a higher-priority polygon.
cm.add_polygon(domain, zone_id=1, z_order=0)
cm.add_polygon(zone2_with_hole, zone_id=2, z_order=1)
cm.add_polygon(hole_polygon, zone_id=3, z_order=2)
```


In [ ]:
donut_case = run_case(
    name="donut-with-overlapping-hole",
    domain_spec={
        "geometry": Polygon([(0, 0), (20, 0), (20, 20), (0, 20)]),
        "zone_id": 1,
        "resolution": 5.0,
        "z_order": 0,
        "dist_max": 25.0,
    },
    polygon_specs=[
        {
            "geometry": Polygon(
                [(5, 5), (15, 5), (15, 15), (5, 15)],
                [[(8, 8), (12, 8), (12, 12), (8, 12)]],
            ),
            "zone_id": 2,
            "resolution": 2.0,
            "z_order": 1,
            "dist_max": 10.0,
        }
    ],
    background_lc=5.0,
)

if donut_case["grid"] is not None and not donut_case["grid"].empty:
    print("Zone assignment in grid:")
    print(donut_case["grid"]["zone_id"].value_counts().to_string())
    print()
    print("? Cells inside the hole (the 4?4 m interior square) are assigned to zone 1.")
    print("  A hole means 'not zone 2'; it is not automatically an empty mesh void.")


In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

if donut_case["grid"] is not None and not donut_case["grid"].empty:
    donut_case["grid"].plot(
        ax=ax, column="zone_id", cmap="Set1", alpha=0.6,
        edgecolor="black", linewidth=0.3,
        categorical=True, legend=True,
    )

# Draw the intended hole boundary for reference
hole_ring = Polygon([(8, 8), (12, 8), (12, 12), (8, 12)])
gpd.GeoSeries([hole_ring], crs=CRS).boundary.plot(
    ax=ax, color="red", linewidth=2, linestyle="--", label="Intended hole boundary"
)

ax.set_title(
    "Limitation 1: hole inside zone 2 polygon\n"
    "Red dashed line = intended hole — interior cells belong to zone 1 (not removed)",
    fontsize=10,
)
ax.set_aspect("equal")
ax.grid(alpha=0.2)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()


---
### Limitation 2 — Connectivity Tolerance Is a Blanket Setting

**What you need to know:** `connectivity_tolerance` is applied globally to every feature — it is not selective. When you raise it to 1.0 m to heal a 0.0005 m gap at one boundary, that same 1.0 m radius is applied to every endpoint everywhere. Any feature endpoint within 1.0 m of the domain boundary (or another nearby feature) will also be snapped, whether you intended it to move or not.

**How to stay safe:** After calling `cm.generate()`, always inspect `clean_points` and `clean_lines` to confirm features are where you expect:

```python
clean_polys, clean_lines, clean_points = cm.generate()
print(clean_points[["point_id", "geometry"]])   # verify positions
```

The cell below shows a 100 m × 2 m domain (very thin) with a centreline and two monitoring points 0.1 m above and below the line. With `connectivity_tolerance=1.0`, the snapping routine processes both points (they are within 1.0 m of the line), but since they are interior to the domain they stay put. In a real workflow with features near the domain boundary, the same tolerance can pull them to unexpected positions — always verify.

In [ ]:

aggressive_connectivity = run_case(
    name="aggressive-connectivity-collapse",
    domain_spec={"geometry": box(0, -1, 100, 1), "zone_id": 1, "resolution": 2.0},
    line_specs=[
        {"geometry": LineString([(0, 0), (100, 0)]), "line_id": "centerline", "resolution": 0.5, "densify": False}
    ],
    point_specs=[
        {"geometry": Point(50,  0.1), "point_id": "above", "resolution": 0.2},
        {"geometry": Point(50, -0.1), "point_id": "below", "resolution": 0.2},
    ],
    background_lc=2.0,
    cm_kwargs={"connectivity_tolerance": 1.0},
)

clean_pts = aggressive_connectivity["clean_points"]
print(f"Input:              2 distinct points  (50, +0.1) and (50, -0.1)")
if clean_pts is not None and not clean_pts.empty:
    for _, row in clean_pts.iterrows():
        print(f"  Clean point coords: ({row.geometry.x:.4f}, {row.geometry.y:.4f})")
print()
print("→ Both points STAYED at their original positions — the centreline has no vertex at x=50")
print("  so there is nothing to snap to at that location.")
print()
print("  In a real workflow, features near the domain boundary or near another feature's")
print("  vertex CAN be pulled unexpectedly. Always inspect clean_points/clean_lines")
print("  after cm.generate() to confirm feature positions.")

print("\nVerification pattern — inspect clean geometry:")
if clean_pts is not None:
    print(clean_pts[["point_id", "geometry"]].to_string(index=True))


In [ ]:
clean_pts = aggressive_connectivity["clean_points"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax_raw, ax_clean = axes

# Zoom tight around the points so the y=±0.1 offset is visible
xlim, ylim = (44, 58), (-0.5, 0.5)

for ax, pts_gdf, pt_color, title in [
    (ax_raw,   aggressive_connectivity["raw_points"],  "red",   "Input: 2 monitoring points\n(0.1 m above and 0.1 m below the centreline)"),
    (ax_clean, clean_pts,                              "green", "After preprocessing (connectivity_tolerance=1.0)\nBoth points stayed at original positions ✓"),
]:
    aggressive_connectivity["raw_polygons"].boundary.plot(ax=ax, color="black", linewidth=0.6)
    aggressive_connectivity["raw_lines"].plot(ax=ax, color="steelblue", linewidth=2, zorder=3)

    if pts_gdf is not None and not pts_gdf.empty:
        pts_gdf.plot(ax=ax, color=pt_color, markersize=60, zorder=5)
        # stagger annotations: first point up, second down
        offsets = [(2, 0.15), (2, -0.20)]
        for i, (_, row) in enumerate(pts_gdf.iterrows()):
            dx, dy = offsets[i]
            ax.annotate(
                f"{row.geometry.y:+.1f} m",
                xy=(row.geometry.x, row.geometry.y),
                xytext=(row.geometry.x + dx, row.geometry.y + dy),
                fontsize=9, color=pt_color,
                arrowprops=dict(arrowstyle="->", color=pt_color, lw=1.2),
            )

    ax.set_title(title, fontsize=10)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect("equal")
    ax.grid(alpha=0.2)
    ax.axhline(0, color="steelblue", linewidth=0.5, linestyle="--", alpha=0.5)

ax_raw.set_facecolor("#fff8f8")
ax_clean.set_facecolor("#f8fff8")

plt.suptitle(
    "Limitation 2: connectivity_tolerance is global\n"
    "Always inspect clean_points / clean_lines after cm.generate() to verify positions",
    fontweight="bold", fontsize=11,
)
plt.tight_layout()
plt.show()

# Practical verification pattern
print("Verification pattern — inspect clean geometry:")
if clean_pts is not None and not clean_pts.empty:
    print(clean_pts[["point_id", "geometry"]].to_string())

---
## Quick Reference

| Problem | Symptom | Fix | API |
|---------|---------|-----|-----|
| Duplicate / near-duplicate vertices | Gmsh would reject degenerate zero-length segments | Handled during robust geometry transfer before OCC creation; inspect short-segment diagnostics, not post-densification vertex counts | `MeshGenerator` geometry transfer |
| Feature doesn't reach domain boundary | Feature is removed from `clean_points` or clipped from `clean_lines`, so it has no local influence | Snap within `N` units, but only to an existing vertex | `ConceptualMesh(connectivity_tolerance=N)` |
| No snap vertex exists where you need one | Large tolerance still does not connect the feature | Resample the reference geometry before adding it | `resample_geometry(geom, spacing)` |
| Irregular source spacing | Local quality diagnostics improve after adding/redistributing vertices | Use the least intrusive spacing tool that improves metrics | `densify=spacing`, `resample_geometry()`, or `simplify_tolerance` + `densify` |
| Field-only refinement line under-refines | Refinement coverage is sparse even though the line is present | Increase DistanceField sampling when the field is under-sampled; fix topology first if the feature is misplaced or clipped | `ThresholdField(..., sampling=N)` |
| Over-digitised / noisy boundaries | Too many redundant vertices drive unnecessary detail | Remove redundant vertices while preserving overall shape | `add_polygon(..., simplify_tolerance=X)` or `add_line(..., simplify_tolerance=X)` |
| Narrow zone + barrier / sliver topology causes Gmsh issues | `generate()` returns `False`, raises, or constraints disappear after healing | Inspect clean geometry and embedding diagnostics first; use healing/tolerance changes only as measured tradeoffs | `MeshGenerator(diagnose=True, heal_shapes=True, tolerance_initial_delaunay=...)` |
| Polygon hole overlapping another zone | Hole is assigned to the lower-priority zone, but user intent may differ | Keep as-is for lower-zone holes; subtract the hole from the domain for an empty void; add a new polygon for a separate zone | `domain.difference(hole)` or `add_polygon(hole, z_order=...)` |
| `connectivity_tolerance` too large | Closely-spaced features can move unexpectedly | Lower tolerance to match the gap size only | Inspect `clean_points` / `clean_lines` after `cm.generate()` |
